# 01 · Tokenization: BPE from scratch and tiktoken

After this notebook you can explain why LLMs read *tokens* (not characters or words), implement byte-pair encoding (BPE) from scratch in an interview, count tokens with `tiktoken` exactly as OpenAI bills them, and explain why the same sentence can cost 4x more tokens in Hindi than in English on GPT-4's tokenizer.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min (tiktoken downloads its encoding files once, then caches them) · **Needs:** basic Python ([strings notebook](../01_python_basics/02_strings_and_fstrings.ipynb))

## Why this matters in interviews

- Everything is priced, limited and measured in tokens: cost per request, context window, rate limits (TPM), latency (tokens per second). You cannot reason about any of them without knowing what a token is.
- "Implement BPE" and "why subwords?" are classic screening questions; the merge loop fits in 20 lines.
- Tokenizer quirks explain real bugs: models that miscount letters in "strawberry", odd arithmetic on long numbers, a 3x bill for non-English users, garbled characters in a streaming UI.
- Self-hosting a model means getting special tokens and the chat template exactly right, or quality silently drops.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo02` | What is a token, and why does everything in this field get measured in tokens? |
| `fo03` | Why do LLMs use a subword tokenizer instead of treating every word as one token? |
| `fo05` | What is the context window, and what actually happens when you exceed it? |
| `fo21` | What is an LLM API call actually made of, and what are you charged for? |
| `in18` | How do you calculate cost per request, properly? |
| `tn24` | What is a chat template and why does it matter when self-hosting? |

In [ ]:
import re
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(0)
np.random.seed(0)
pd.set_option("display.width", 120)
print("ready")

## 1. Why tokens, and not characters or words?

**In plain English:** a neural network only understands numbers, so text must be chopped into pieces and each piece mapped to an integer id from a fixed list (the *vocabulary*). The question is how big the pieces should be.

| Unit | Vocabulary size | Sequence length | Unknown words? | Problem |
|---|---|---|---|---|
| **Character** | tiny (~100-300) | very long (1 token per char) | never | attention cost grows with n², so long sequences are expensive; each token carries little meaning |
| **Word** | huge (millions: every name, typo, inflection, language) | short | constantly (`[UNK]`) | "unhappiness", "GPT-5o", "Sujan" become `[UNK]`; the embedding table explodes |
| **Subword (BPE, WordPiece, Unigram)** | chosen: 32k-200k | short-ish (~4 English chars per token) | never, with a byte-level fallback | a few quirks (below) |

Subwords are the compromise: frequent words become one token (`" the"`), rare words are spelled from pieces (`"un" + "happiness"`), and anything else falls back to raw bytes. Let us measure the first two options on a small corpus.

In [ ]:
CORPUS = """Language models read text as a sequence of tokens. A token is a small piece of text: often a whole
common word, sometimes part of a longer word, sometimes a single character or byte. The model never sees
letters directly; it sees token ids, which are integers that index a table of learned vectors.
Tokenizers are trained before the model is trained. The training data decides which pieces become tokens,
so a tokenizer trained mostly on English text is very good at English and much worse at other languages.
Common words such as the, and, of, to, in and is become single tokens. Longer and rarer words are split
into pieces that appeared often enough in the training data, like token, izer and ization.
When you send a request to a model, you pay for the tokens in the prompt and the tokens in the answer.
The context window is also counted in tokens, so the tokenizer decides how much text fits in a request.
Counting tokens correctly is the first step in estimating cost, latency and whether a document fits.
The tokenizer is the part of the system that people forget, and it is the part that explains many bugs.
A model that cannot count the letters in a word is usually not stupid; it simply never saw the letters.
Numbers are split into chunks of digits, so arithmetic on long numbers is harder than it looks.
Spaces are usually attached to the front of the next word, which is why the space matters so much.
Training a tokenizer means counting which neighbouring pieces appear together most often, merging the
most frequent pair into a new piece, and repeating this thousands of times until the vocabulary is full.
That algorithm is called byte pair encoding, and it is used by the tokenizers of most modern models."""

HELD_OUT = ("Tokenization splits the incoming prompt into pieces before the model reads it. "
            "Unhappiness, tokenizers and retokenization are handled without any unknown words.")

def words_of(text):
    return re.findall(r"[A-Za-z]+|[0-9]+|[^\sA-Za-z0-9]", text)

char_vocab = sorted(set(CORPUS))
word_vocab = set(words_of(CORPUS))
held_words = words_of(HELD_OUT)
oov = [w for w in held_words if w not in word_vocab]

print(f"corpus: {len(CORPUS)} characters")
print(f"character-level: vocab {len(char_vocab):4d}, held-out sequence length {len(HELD_OUT)}")
print(f"word-level     : vocab {len(word_vocab):4d}, held-out sequence length {len(held_words)}")
print(f"word-level unknown words in held-out text ({len(oov)}): {oov}")

Characters give no unknowns but long sequences; words give short sequences but many `[UNK]`s even for simple English ("splits", "handled", "Unhappiness"). A real word vocabulary would need millions of entries to cover names, typos, code and other languages. **Byte-pair encoding** fixes both.

## 2. Byte-pair encoding (BPE) from scratch

**In plain English:** start with the smallest pieces (bytes). Count every pair of neighbouring pieces in the training text. Glue the most frequent pair into one new piece. Repeat. Each glue step ("merge") adds one entry to the vocabulary. To encode new text, replay the merges in the order they were learned.

First on a classic toy string, so every step is visible (letters instead of bytes for readability):

In [ ]:
def pair_counts(seq):
    return Counter(zip(seq, seq[1:]))

def merge_pair(seq, pair, new_symbol):
    """Replace every non-overlapping occurrence of `pair` (left to right) with `new_symbol`."""
    out, i = [], 0
    while i < len(seq):
        if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
            out.append(new_symbol); i += 2
        else:
            out.append(seq[i]); i += 1
    return out

toy = list("aaabdaaabac")
print("start      :", toy)
for step, new_symbol in enumerate("ZYX", start=1):
    counts = pair_counts(toy)
    best = max(counts, key=counts.get)            # ties -> first seen (deterministic)
    toy = merge_pair(toy, best, new_symbol)
    print(f"merge {step}: {best} (seen {counts[best]}x) -> {new_symbol!r}:", toy)

Eleven symbols became five with a 3-entry merge table. Real tokenizers do exactly this, with three changes:

1. **Bytes, not letters.** The starting vocabulary is the 256 possible byte values, so *any* string (Hindi, emoji, binary junk) can be encoded. There is never an unknown token. This is *byte-level BPE* (GPT-2, GPT-4, Llama 3).
2. **Pre-tokenization.** Text is first split into word-like chunks with a regex (a space stays attached to the front of the following word), and merges never cross chunk boundaries. This stops junk like `"dog."` or `"the cat"` from becoming tokens.
3. **Scale.** ~50k-200k merges learned on terabytes of text, not 3 on a toy string.

### 2.1 Train a byte-level BPE on our small corpus

In [ ]:
# Pre-tokenizer: optional leading space + letters | + digits | + punctuation run | whitespace run.
# Every character matches one branch, so joining the chunks gives back the original text exactly.
PRETOKENIZE = re.compile(r" ?[A-Za-z]+| ?[0-9]+| ?[^A-Za-z0-9\s]+|\s+")
assert "".join(PRETOKENIZE.findall(CORPUS)) == CORPUS

def train_bpe(text, n_merges):
    """Byte-level BPE. Returns merges {(id_a, id_b): new_id} in learned order, and vocab {id: bytes}."""
    # Speed trick: train on *unique* chunks weighted by frequency, not on the raw stream.
    chunk_freq = Counter(PRETOKENIZE.findall(text))
    chunks = {tuple(c.encode("utf-8")): f for c, f in chunk_freq.items()}
    vocab = {i: bytes([i]) for i in range(256)}
    merges = {}
    for k in range(n_merges):
        counts = Counter()
        for ids, f in chunks.items():
            for pair in zip(ids, ids[1:]):
                counts[pair] += f
        if not counts:
            break
        best = max(counts, key=counts.get)
        new_id = 256 + k
        merges[best] = new_id
        vocab[new_id] = vocab[best[0]] + vocab[best[1]]
        chunks = {tuple(merge_pair(list(ids), best, new_id)): f for ids, f in chunks.items()}
    return merges, vocab

t0 = time.perf_counter()
merges, vocab = train_bpe(CORPUS, n_merges=200)
print(f"learned {len(merges)} merges in {time.perf_counter() - t0:.2f}s -> vocab size {len(vocab)} (256 bytes + merges)")

In [ ]:
def show(b):
    return repr(b.decode("utf-8", errors="replace"))

rows = [(new_id, show(vocab[a]), show(vocab[b]), show(vocab[new_id])) for (a, b), new_id in merges.items()]
merge_table = pd.DataFrame(rows, columns=["new id", "left", "right", "merged token"])
print("first 15 merges (the most frequent pairs in the corpus):")
print(merge_table.head(15).to_string(index=False))
print("\nthe longest tokens learned:")
longest = merge_table.assign(chars=merge_table["merged token"].str.len() - 2).nlargest(6, "chars")
print(longest.drop(columns="chars").to_string(index=False))

Read the table top to bottom: the first merges glue a space to a common first letter (`' t'`, `' a'`) and join frequent letter pairs (`'he'`, `'er'`, `'in'`). Within about ten merges whole frequent words appear, *with their leading space* (`' the'`, `' to'`, `' token'`). Later merges build longer, rarer pieces on top of earlier ones. The merge order *is* the tokenizer: encoding replays the merges by rank.

In [ ]:
def bpe_encode(text, merges):
    ids_out = []
    for chunk in PRETOKENIZE.findall(text):
        ids = list(chunk.encode("utf-8"))
        while len(ids) >= 2:
            # apply the earliest-learned merge that exists in this chunk
            pair = min(zip(ids, ids[1:]), key=lambda p: merges.get(p, float("inf")))
            if pair not in merges:
                break
            ids = merge_pair(ids, pair, merges[pair])
        ids_out.extend(ids)
    return ids_out

def bpe_decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

def pieces(ids, vocab):
    return [vocab[i].decode("utf-8", errors="replace") for i in ids]

ids = bpe_encode(HELD_OUT, merges)
print("held-out text as our tokens:")
print("|".join(pieces(ids, vocab)))
print(f"\n{len(HELD_OUT)} characters -> {len(ids)} tokens ({len(HELD_OUT) / len(ids):.2f} chars per token)")

for text in [HELD_OUT, "Supercalifragilistic tokenizers!", "नमस्ते 🙂 ok", "x = 3.14159\n"]:
    assert bpe_decode(bpe_encode(text, merges), vocab) == text, text
print("round-trip decode(encode(x)) == x holds for English, an unseen word, Hindi + emoji, and code")

No unknown tokens anywhere: "Unhappiness" is spelled from pieces the corpus taught us, and Hindi or emoji (never seen in training) fall back to their raw UTF-8 bytes. That fallback is why byte-level BPE is the default for modern LLMs.

### 2.2 The core trade-off: vocabulary size vs sequence length

More merges means a bigger vocabulary (a bigger embedding table and output softmax) but shorter sequences (cheaper attention, more text per context window). We can replay only the first *k* merges to see the curve.

In [ ]:
ks = [0, 10, 25, 50, 75, 100, 150, 200]
train_len, held_len = [], []
for k in ks:
    first_k = {p: i for p, i in merges.items() if i < 256 + k}
    train_len.append(len(bpe_encode(CORPUS, first_k)))
    held_len.append(len(bpe_encode(HELD_OUT, first_k)))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot([256 + k for k in ks], np.array(train_len) / len(CORPUS), "o-", label="training corpus")
ax.plot([256 + k for k in ks], np.array(held_len) / len(HELD_OUT), "s--", label="held-out text")
ax.set_xlabel("vocabulary size (256 bytes + merges)")
ax.set_ylabel("tokens per character")
ax.set_title("More merges -> shorter sequences, with diminishing returns")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

assert train_len == sorted(train_len, reverse=True), "each merge can only shorten the training text"
print(f"training corpus: {train_len[0]} -> {train_len[-1]} tokens;  held-out: {held_len[0]} -> {held_len[-1]} tokens")

Two lessons in one chart. The curve flattens: the 200th merge saves far fewer tokens than the 1st. And the held-out line sits above the training line: merges learned on one kind of text compress *that* text best. This is exactly why GPT-4's English-heavy tokenizer is cheap for English and expensive for Hindi (section 4).

| Tokenizer | Vocab | Used by |
|---|---|---|
| GPT-2 BPE | 50,257 | GPT-2, GPT-3 |
| `cl100k_base` | ~100k | GPT-4, GPT-3.5-turbo, `text-embedding-3-*` |
| `o200k_base` | ~200k | GPT-4o, GPT-4.1, o-series, GPT-5 |
| Llama 3 (tiktoken-style BPE) | 128k | Llama 3.x |
| WordPiece | 30,522 | BERT (encoder models) |
| SentencePiece Unigram / BPE | 32k-256k | T5, Llama 2, Gemma |

## 3. Why byte-level means "no unknown tokens" (and why other scripts cost more)

In [ ]:
for ch in ["a", "é", "न", "日", "🙂"]:
    b = ch.encode("utf-8")
    print(f"{ch!r}: {len(b)} UTF-8 byte(s) {list(b)}")

hindi = "नमस्ते"
print(f"\nour English-trained BPE on {hindi!r}: {len(hindi)} characters -> {len(bpe_encode(hindi, merges))} tokens (all raw bytes)")

English letters are 1 byte, Devanagari and Japanese characters are 3 bytes, emoji are 4. A tokenizer that learned few merges for a script falls back towards one token *per byte*, so a Hindi word can cost several times more tokens than its English translation. Bigger, more multilingual vocabularies (`o200k_base`) learn merges for those scripts too.

## 4. tiktoken: the tokenizers OpenAI actually bills with

`tiktoken` is OpenAI's fast BPE library (Rust core). The first `get_encoding` call downloads the merge file (a few MB) and caches it; afterwards it works offline. If it cannot download, we fall back to our own BPE so the rest of the notebook still runs.

In [ ]:
import tiktoken

class OurBPE:
    """Our from-scratch tokenizer behind the same .encode/.decode interface as tiktoken."""
    name, n_vocab = "our_bpe", len(vocab)
    def encode(self, text, **_):
        return bpe_encode(text, merges)
    def decode(self, ids):
        return bpe_decode(ids, vocab)
    def decode_single_token_bytes(self, i):
        return vocab[i]

encoders = {"our_bpe": OurBPE()}
for name in ["cl100k_base", "o200k_base"]:
    try:
        encoders[name] = tiktoken.get_encoding(name)
    except Exception as e:  # no network on first use, blocked download, ...
        print(f"could not load {name} ({type(e).__name__}: {str(e)[:80]}) - using our BPE in its place")
        encoders[name] = encoders["our_bpe"]

for name, enc in encoders.items():
    print(f"{name:12s} vocab size {enc.n_vocab:,}")
print("tiktoken.encoding_for_model('gpt-4o') ->", tiktoken.encoding_for_model("gpt-4o").name)

In [ ]:
def token_pieces(enc, text):
    return [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in enc.encode(text)]

sentence = "Tokenizers split text into pieces; the model only sees integer ids."
for name, enc in encoders.items():
    ids = enc.encode(sentence)
    print(f"{name:12s} {len(ids):2d} tokens: " + "|".join(token_pieces(enc, sentence)))
print("\ncl100k ids:", encoders["cl100k_base"].encode(sentence))

The real tokenizers were trained on vastly more text, so they keep whole words like `" integer"` and `" only"` as single tokens where our 200-merge toy still spells them out (`in|te|g|er`). The rule of thumb for English: **1 token ≈ 4 characters ≈ 0.75 words**.

## 5. Same meaning, different price: tokens per language and for code

In [ ]:
TEXTS = {
    "English":  "The weather is nice today, so let's go for a walk in the park and then have lunch together.",
    "Spanish":  "Hoy hace buen tiempo, así que vamos a dar un paseo por el parque y luego almorzar juntos.",
    "Hindi":    "आज मौसम अच्छा है, तो चलो पार्क में टहलने चलते हैं और फिर साथ में दोपहर का खाना खाते हैं।",
    "Japanese": "今日は天気がいいので、公園を散歩してから一緒にお昼ご飯を食べましょう。",
    "Python":   "def mean(xs):\n    total = sum(xs)\n    return total / len(xs) if xs else 0.0\n",
}
counts = pd.DataFrame({name: {lang: len(enc.encode(t)) for lang, t in TEXTS.items()} for name, enc in encoders.items()})
counts["chars"] = [len(t) for t in TEXTS.values()]
counts["cl100k vs English"] = (counts["cl100k_base"] / counts.loc["English", "cl100k_base"]).round(2)
print(counts.to_string())

In [ ]:
langs = list(TEXTS)
x = np.arange(len(langs))
width = 0.27
fig, ax = plt.subplots(figsize=(9, 3.8))
for j, name in enumerate(encoders):
    ax.bar(x + (j - 1) * width, counts[name].values, width, label=name)
ax.set_xticks(x, langs)
ax.set_ylabel("tokens")
ax.set_title("Token count for (roughly) the same sentence, by tokenizer")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.show()

cl = counts["cl100k_base"]
if encoders["cl100k_base"] is not encoders["our_bpe"]:
    assert cl["Hindi"] > 1.5 * cl["English"], "Hindi should cost well over English on cl100k_base"
    assert counts.loc["Hindi", "o200k_base"] < cl["Hindi"], "o200k_base's bigger vocab should help Hindi"
print(f"Hindi costs {cl['Hindi'] / cl['English']:.1f}x the English tokens on cl100k_base, "
      f"{counts.loc['Hindi', 'o200k_base'] / counts.loc['English', 'o200k_base']:.1f}x on o200k_base")

What this means in practice:

- **Cost:** you pay per token, so a Hindi or Japanese product can cost several times more per conversation than the English one - on the same model, for the same meaning.
- **Context window:** a 128k-token window holds far fewer Hindi words than English words. RAG chunk sizes should be set in tokens, not characters.
- **Latency:** output speed is in tokens/second, so more tokens per word means slower-looking answers.
- **Tokenizer upgrades matter:** on this sentence `o200k_base` needs about a third of `cl100k_base`'s tokens for Hindi, bringing it close to English. Our English-only toy BPE is the extreme case: it spells Hindi and Japanese nearly byte by byte.

### A cost estimate from token counts

Prices below are **illustrative** (gpt-4.1-mini list prices at the time of writing) - always check the provider's pricing page. The method is what interviewers want: *tokens in × input price + tokens out × output price*, per request, times volume.

In [ ]:
PRICE_PER_1M = {"input": 0.40, "output": 1.60}   # USD per 1M tokens, illustrative

def cost_usd(prompt_tokens, output_tokens, price=PRICE_PER_1M):
    return (prompt_tokens * price["input"] + output_tokens * price["output"]) / 1e6

enc = encoders["o200k_base"]
system_prompt = "You are a helpful travel assistant. Answer in the user's language, in at most three sentences."
rows = []
for lang in ["English", "Spanish", "Hindi", "Japanese"]:
    prompt = len(enc.encode(system_prompt)) + len(enc.encode(TEXTS[lang])) * 10   # ~10 sentences of user text
    output = len(enc.encode(TEXTS[lang])) * 3                                      # ~3 sentences of reply
    per_req = cost_usd(prompt, output)
    rows.append((lang, prompt, output, per_req * 1_000_000))
table = pd.DataFrame(rows, columns=["language", "prompt tok", "output tok", "$ per 1M requests"])
table["x English"] = table["$ per 1M requests"] / table.loc[0, "$ per 1M requests"]
print(table.round(2).to_string(index=False))

Even on the multilingual `o200k_base`, Hindi and Japanese cost more per request; on a `cl100k_base`-era model the gap would be the ~2-4x from the chart. Notice also that output tokens are priced 4x input here, so a verbose answer can cost more than a long prompt. Real cost models also include cached-input discounts, retries, tool-call round trips and reasoning tokens (billed as output even though you never see them).

## 6. Tokenizer quirks interviewers love

In [ ]:
enc = encoders["cl100k_base"]
def show_tokens(text):
    p = token_pieces(enc, text)
    print(f"{text!r:28} {len(p)} tok: " + " | ".join(repr(s) for s in p))

print("-- leading spaces and case change the token:")
for t in ["hello", " hello", "Hello", "HELLO", "  hello"]:
    show_tokens(t)
print("-- numbers are chunked in groups of up to 3 digits (left to right):")
for t in ["7", "1234567", "12345678", "3.14159", "2024-09-29"]:
    show_tokens(t)

- `" hello"` (with a space) and `"hello"` are **different tokens with different ids**. A trailing space at the end of a prompt can hurt completions, because the model normally expects the space at the *start* of the next token.
- `"1234567"` is `123|456|7`: digit groups do not line up with place value, one reason arithmetic on long numbers is unreliable (tools or code execution are the fix).

In [ ]:
for t in ["strawberry", " strawberry", "How many r's are in strawberry?"]:
    show_tokens(t)

word_tokens = token_pieces(enc, "strawberry")
assert len(word_tokens) > 1 and "r" not in word_tokens, "the letters are hidden inside multi-letter tokens"
print(f"\nactual count of 'r' in strawberry: {'strawberry'.count('r')}")

The model receives `str|aw|berry` (or a single `" strawberry"` token when there is a leading space) - three or one opaque ids, not ten letters. It can only "count the r's" if it memorised facts about how those tokens are spelled. That is why letter-counting and reversing strings are weak spots, and why "spell it out letter by letter first" helps.

**Streaming gotcha:** a token can be *part* of a UTF-8 character. Decode token-by-token and you print `�`; buffer the bytes and decode when a character is complete.

In [ ]:
text = "नमस्ते"
ids = enc.encode(text)
naive = "".join(enc.decode([i]) for i in ids)        # decode each token separately (a naive streaming UI)
buffered = b"".join(enc.decode_single_token_bytes(i) for i in ids).decode("utf-8")
print(f"{len(ids)} tokens for {len(text)} characters")
print("decode token by token:", naive)
print("buffer bytes, then decode:", buffered)
if enc is not encoders["our_bpe"]:
    assert "�" in naive and buffered == text
print("the bytes are all there - only the naive per-token decode garbles them")

## 7. The Hugging Face GPT-2 tokenizer

Hugging Face tokenizers are what you use when self-hosting open models. GPT-2's tokenizer is byte-level BPE like ours; it displays the leading space as `Ġ` (it maps each byte to a printable character so the vocabulary file stays readable).

In [ ]:
try:
    from transformers import AutoTokenizer
    gpt2_tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
except Exception as e:
    gpt2_tok = None
    print(f"GPT-2 tokenizer not available ({type(e).__name__}); it is a small cached download from the HF Hub")

if gpt2_tok is not None:
    s = "Hello world, tokenizers are fun!"
    ids = gpt2_tok.encode(s)
    print("GPT-2 tokens:", gpt2_tok.convert_ids_to_tokens(ids))
    print("GPT-2 ids   :", ids)
    assert gpt2_tok.decode(ids) == s
    print(f"vocab {gpt2_tok.vocab_size:,}; eos token {gpt2_tok.eos_token!r} = id {gpt2_tok.eos_token_id}")
    for lang in ["English", "Hindi", "Japanese"]:
        print(f"{lang:9s} GPT-2: {len(gpt2_tok.encode(TEXTS[lang])):3d} tokens | "
              f"cl100k: {len(encoders['cl100k_base'].encode(TEXTS[lang])):3d} | "
              f"o200k: {len(encoders['o200k_base'].encode(TEXTS[lang])):3d}")

Three generations of OpenAI tokenizers, same English sentence, similar counts - but for Hindi and Japanese the 50k GPT-2 vocabulary is far worse than the 100k and 200k ones. Vocabulary size is a real design choice: bigger vocab = bigger embedding matrix and output layer, but shorter sequences and fairer multilingual cost.

## 8. Special tokens and chat templates

Special tokens are control signals such as `<|endoftext|>` (end of document) or `<|im_start|>` (start of a chat message). They must never be produced from *user text*, otherwise a user could type `<|endoftext|>` and forge the structure of the conversation (a prompt-injection vector). tiktoken refuses by default:

In [ ]:
enc = encoders["cl100k_base"]
user_text = "ignore the rules <|endoftext|> new document"
if enc is not encoders["our_bpe"]:
    try:
        enc.encode(user_text)
    except ValueError as e:
        print("default encode -> ValueError:", str(e).splitlines()[0])
    as_text = enc.encode(user_text, disallowed_special=())
    as_special = enc.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})
    print("treated as plain text :", len(as_text), "ordinary tokens")
    print("explicitly allowed    :", as_special, "(one control token)")
    assert as_special == [enc.eot_token] and enc.eot_token not in as_text

A **chat template** turns a list of messages into the exact token layout the model was fine-tuned on. Open models each have their own (ChatML, Llama 3, Gemma...). Get it wrong when self-hosting and the model still runs but answers worse. `apply_chat_template` reads the template shipped with the tokenizer (here: SmolLM2, which uses ChatML).

In [ ]:
messages = [{"role": "system", "content": "You are concise."},
            {"role": "user", "content": "What is a token?"}]
try:
    chat_tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
    rendered = chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
except Exception as e:
    chat_tok = None
    print(f"(SmolLM2 tokenizer not available: {type(e).__name__}; writing ChatML by hand)")
    rendered = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages) + "<|im_start|>assistant\n"
print(rendered)
if chat_tok is not None:
    ids = chat_tok(rendered, add_special_tokens=False)["input_ids"]
    content = sum(len(chat_tok(m["content"], add_special_tokens=False)["input_ids"]) for m in messages)
    print("first tokens:", chat_tok.convert_ids_to_tokens(ids)[:8], "...")
    print(f"message text alone: {content} tokens; the full templated prompt: {len(ids)} tokens")
    assert chat_tok.convert_ids_to_tokens(ids)[0] == "<|im_start|>", "a special token is ONE token"

`<|im_start|>` is a single special token for this tokenizer, and every message carries a few tokens of role markers and newlines; the prompt ends by priming the reply with `assistant`. That is why your count of the message text is always below the API's `usage.prompt_tokens`. OpenAI's own recipe counts ~3 extra tokens per message plus 3 for the reply priming, which is close enough for budgeting (exact for older models; always trust `usage` for billing).

## Try it yourself

**1.** Write `num_tokens_from_messages(messages)` that estimates prompt tokens for a chat request: `o200k_base` tokens of each message's content + 3 per message + 3 for the reply priming. Compare with the plain content count.

In [ ]:
# Solution — try it yourself first, then run this
def num_tokens_from_messages(messages, enc=encoders["o200k_base"], per_message=3, reply_priming=3):
    return sum(per_message + len(enc.encode(m["content"])) for m in messages) + reply_priming

content_tokens = sum(len(encoders["o200k_base"].encode(m["content"])) for m in messages)
est = num_tokens_from_messages(messages)
print(f"content only: {content_tokens} tokens, estimated request: {est} tokens")
assert est == content_tokens + 3 * len(messages) + 3

**2.** Train our BPE with only 50 merges and with 200 merges. Which merged tokens does the 200-merge tokenizer have that the 50-merge one does not? Print five of them.

In [ ]:
# Solution — try it yourself first, then run this
m50, v50 = train_bpe(CORPUS, 50)
extra = [vocab[i].decode("utf-8", "replace") for p, i in merges.items() if p not in m50]
print(f"50-merge vocab {len(v50)}, 200-merge vocab {len(vocab)}; examples only in the bigger one: {extra[-5:]}")
assert set(m50).issubset(merges), "training is greedy, so the first 50 merges are identical"

**3.** A customer sends 2,000 support chats a day, each with a 600-token system prompt, 400 tokens of history and a 150-token reply. Using `cost_usd`, what is the monthly (30-day) bill? What is it if the system prompt is cut to 200 tokens?

In [ ]:
# Solution — try it yourself first, then run this
per_day = 2000
before = cost_usd(600 + 400, 150) * per_day * 30
after = cost_usd(200 + 400, 150) * per_day * 30
print(f"monthly: ${before:.2f} -> ${after:.2f}  (saves {100 * (1 - after / before):.0f}%)")

## Say it in an interview

- **30-second answer (what is a token):** "Models map text to integer ids from a fixed vocabulary learned by a tokenizer - usually byte-level BPE. A token is a frequent chunk: whole common words, pieces of rare words, raw bytes as a fallback. English averages ~4 characters or 0.75 words per token. Cost, context windows, rate limits and speed are all counted in tokens, and the tokenizer is fixed at pretraining time."
- **Why subwords:** characters make sequences too long (attention is O(n²)); words need millions of entries and still hit unknowns. Subwords give a bounded vocabulary (32k-200k), no unknowns (byte fallback) and short sequences.
- **BPE in one breath:** start from 256 bytes, count adjacent pairs, merge the most frequent, record the merge, repeat until the vocab size is reached; encode by replaying merges in rank order. WordPiece (BERT) picks merges by likelihood gain; SentencePiece Unigram prunes a big vocab by likelihood.
- **Key numbers:** GPT-2 50,257; cl100k_base ~100k; o200k_base ~200k; Llama 3 128k. In our example Hindi costs ~4x English on cl100k_base and ~1.3x on o200k_base.
- **Traps:** `" hello" != "hello"`; numbers split in 3-digit chunks; letter counting fails because letters are hidden in tokens; per-token decoding breaks multi-byte characters in streaming; user text must never become special tokens; the chat template must match the one the model was trained with.
- **Follow-ups:** "Why does the API count more tokens than my tiktoken count?" (message formatting overhead, tool schemas, images). "Can I change a model's tokenizer?" (not without re-training or at least re-learning embeddings - token ids are baked into the weights).

## Next

- [02 · Embeddings and similarity](02_embeddings_and_similarity.ipynb) - what happens to each token id next: it becomes a vector.
- [HF tokenizers, models and generate](../10_huggingface/02_tokenizers_models_and_generate.ipynb) - tokenizers for open models in depth.
- [Conversation memory and context windows](../08_llm_apps/06_conversation_memory_and_context_windows.ipynb) - token budgets in a real chat app.
- [Chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) - chunk by tokens, not characters.
- Theory: [genai_flow course](../../genai_flow/index.html) (chapter 2 "Tokens" has a live tokenizer) · [interview bank](../../ai_interview_prep/index.html)